In [1]:
#!sudo python ../setup.py develop

In [2]:
from slowfast.config.defaults import assert_and_infer_cfg
from slowfast.utils.parser import load_config, parse_args

C:\Users\dev\anaconda3\envs\slowfast\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import cv2
from IPython.display import clear_output

In [4]:
import time

import numpy as np
import torch

import slowfast.models.optimizer as optim
from slowfast.datasets import loader
from slowfast.models import build_model

import slowfast.utils.checkpoint as cu
import slowfast.utils.distributed as du

from slowfast.utils.meters import AVAMeter, EpochTimer, TrainMeter, ValMeter


In [5]:
import matplotlib.pyplot as plt

In [22]:
import sys
#sys.path.append("/home/developer/SlowFast/tools") for linux
sys.path.append("d:\\works\\SlowFast\\tools") # for windows

In [23]:
sys.path

['D:\\visualplans\\FloorplanTransformation\\visualplan',
 'C:\\Users\\dev\\anaconda3\\envs\\slowfast\\python38.zip',
 'C:\\Users\\dev\\anaconda3\\envs\\slowfast\\DLLs',
 'C:\\Users\\dev\\anaconda3\\envs\\slowfast\\lib',
 'C:\\Users\\dev\\anaconda3\\envs\\slowfast',
 '',
 'C:\\Users\\dev\\anaconda3\\envs\\slowfast\\lib\\site-packages',
 'd:\\works\\detectron2',
 'd:\\works\\slowfast',
 'C:\\Users\\dev\\anaconda3\\envs\\slowfast\\lib\\site-packages\\opencv_python-4.12.0.88-py3.8-win-amd64.egg',
 'C:\\Users\\dev\\anaconda3\\envs\\slowfast\\lib\\site-packages\\win32',
 'C:\\Users\\dev\\anaconda3\\envs\\slowfast\\lib\\site-packages\\win32\\lib',
 'C:\\Users\\dev\\anaconda3\\envs\\slowfast\\lib\\site-packages\\Pythonwin',
 '/home/developer/SlowFast/tools',
 'd:\\works\\SlowFast\\tools']

In [24]:
from train_net import eval_epoch

```
--cfg
./demo/AVA/SLOWFAST_32x2_R101_50_50.yaml
--opts
DATA.PATH_TO_DATA_DIR
"D:\\works\\AVA"
TRAIN.ENABLE
True
NUM_GPUS
1
TRAIN.BATCH_SIZE
8
AVA.TRAIN_LISTS
['train_short.csv']
AVA.TEST_LISTS
['val_short.csv']
AVA.TRAIN_GT_BOX_LISTS
['ava_train_v2.2_short.csv']
AVA.TEST_PREDICT_BOX_LISTS
['ava_val_v2.2_short.csv']
```

In [7]:
sys.argv = [
    "tools/run_net.py",  # argv[0]는 프로그램 이름 (아무거나 넣어도 됨)
    "--cfg", "./AVA/SLOWFAST_32x2_R101_50_50.yaml",
    "--opts",
    "DATA.PATH_TO_DATA_DIR","D:\\works\\AVA",
    "TRAIN.ENABLE","True",
    "NUM_GPUS","1",
    "TRAIN.BATCH_SIZE","8",
    "TRAIN.CHECKPOINT_TYPE",'pytorch',
    "TRAIN.CHECKPOINT_FILE_PATH","../weights/SLOWFAST_32x2_R101_50_50.pkl",
    "AVA.TRAIN_LISTS","['train_short.csv']",
    "AVA.TEST_LISTS", "['val_short.csv']",
    "AVA.TRAIN_GT_BOX_LISTS","['ava_train_v2.2_short.csv']",
    "AVA.TEST_PREDICT_BOX_LISTS", "['ava_val_v2.2_short.csv']"
]
args = parse_args() 

## checkpoint file
checkpoints 폴더에서 donwload `SLOWFAST_32x2_R101_50_50.pkl`
```
wget https://dl.fbaipublicfiles.com/pyslowfast/model_zoo/ava/SLOWFAST_32x2_R101_50_50.pkl
```

In [8]:
path_to_config = args.cfg_files[0]
cfg = load_config(args, path_to_config)
cfg = assert_and_infer_cfg(cfg)

In [9]:
# Set random seed from configs.
np.random.seed(cfg.RNG_SEED)
torch.manual_seed(cfg.RNG_SEED)

In [10]:
# Build the video model and print model statistics.
model = build_model(cfg)

In [11]:
# Construct the optimizer.
optimizer = optim.construct_optimizer(model, cfg)
# Create a GradScaler for mixed precision training
scaler = torch.cuda.amp.GradScaler(enabled=cfg.TRAIN.MIXED_PRECISION)

bn 430, non bn 238, zero 0, no grad 0


In [12]:
cfg.TRAIN.CHECKPOINT_FILE_PATH

'../weights/SLOWFAST_32x2_R101_50_50.pkl'

In [13]:
cfg.TRAIN.CHECKPOINT_TYPE = 'pytorch'

In [14]:
checkpoint_epoch = cu.load_checkpoint(
    cfg.TRAIN.CHECKPOINT_FILE_PATH,
    model,
    cfg.NUM_GPUS > 1,
    optimizer,
    scaler if cfg.TRAIN.MIXED_PRECISION else None,
    inflation=cfg.TRAIN.CHECKPOINT_INFLATE,
    convert_from_caffe2=cfg.TRAIN.CHECKPOINT_TYPE == "caffe2",
    epoch_reset=cfg.TRAIN.CHECKPOINT_EPOCH_RESET,
    clear_name_pattern=cfg.TRAIN.CHECKPOINT_CLEAR_NAME_PATTERN,
    image_init=cfg.TRAIN.CHECKPOINT_IN_INIT,
)
start_epoch = checkpoint_epoch + 1
cur_epoch = start_epoch

missing keys: []
unexpected keys: []


In [15]:
start_epoch

1

In [16]:
val_loader = loader.construct_loader(cfg, "val")
train_loader = loader.construct_loader(cfg, "train")
precise_bn_loader = (
    loader.construct_loader(cfg, "train", is_precise_bn=True)
    if cfg.BN.USE_PRECISE_STATS
    else None
)
val_meter = AVAMeter(len(val_loader), cfg, mode="val")

In [25]:
eval_epoch(
    val_loader,
    model,
    val_meter,
    cur_epoch,
    cfg,
    train_loader,
    None,
)

{ 'PascalBoxes_PerformanceByCategory/AP@0.5IOU/answer phone': 0.0036231884057971015,
  'PascalBoxes_PerformanceByCategory/AP@0.5IOU/bend/bow (at the waist)': 0.0073992956405863316,
  'PascalBoxes_PerformanceByCategory/AP@0.5IOU/carry/hold (an object)': 0.011179133875831231,
  'PascalBoxes_PerformanceByCategory/AP@0.5IOU/climb (e.g., a mountain)': 0.0,
  'PascalBoxes_PerformanceByCategory/AP@0.5IOU/close (e.g., a door, a box)': 0.007716049382716049,
  'PascalBoxes_PerformanceByCategory/AP@0.5IOU/crouch/kneel': 0.027477123071143528,
  'PascalBoxes_PerformanceByCategory/AP@0.5IOU/cut': 0.0,
  'PascalBoxes_PerformanceByCategory/AP@0.5IOU/dance': 0.0,
  'PascalBoxes_PerformanceByCategory/AP@0.5IOU/dress/put on clothing': 0.03293807641633729,
  'PascalBoxes_PerformanceByCategory/AP@0.5IOU/drink': 0.0,
  'PascalBoxes_PerformanceByCategory/AP@0.5IOU/drive (e.g., a car, a truck)': 0.10566398839491493,
  'PascalBoxes_PerformanceByCategory/AP@0.5IOU/eat': 0.0,
  'PascalBoxes_PerformanceByCategory